# Tutorial: Interacting with Google Cloud Storage using `storage.py`

This notebook provides a guided tutorial on how to use the `storage.py` Python module, which offers a convenient wrapper for common Google Cloud Storage (GCS) operations. It simplifies tasks like uploading, downloading, and listing objects within a specified GCS bucket.

## 1. Setup and Authentication

First, we need to install the `google-cloud-storage` library, which `storage.py` depends on. If you're running this in Google Colab, you'll also need to authenticate your Google account to access GCS.

In [ ]:
# Install the necessary library
!pip install google-cloud-storage

# Authenticate with Google Cloud (for Colab environments)
# If you are running this locally, ensure your gcloud CLI is authenticated 
# or your GOOGLE_APPLICATION_CREDENTIALS environment variable is set.
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Google Colab authentication successful.")
except ImportError:
    print("Not in Colab environment or google.colab.auth not found. Assuming local authentication.")


### Define Project and Bucket Information

You'll need a Google Cloud Project ID and a GCS bucket name. If you don't have a bucket, you can create one in the [GCP Console](https://console.cloud.google.com/storage/buckets).

**Important:** Replace `YOUR_PROJECT_ID` and `YOUR_GCS_BUCKET_NAME` with your actual project ID and bucket name.

In [ ]:
PROJECT_ID = "YOUR_PROJECT_ID"  # e.g., "my-gcp-project-12345"
BUCKET_NAME = "YOUR_GCS_BUCKET_NAME" # e.g., "my-unique-bucket-for-demo"

if PROJECT_ID == "YOUR_PROJECT_ID" or BUCKET_NAME == "YOUR_GCS_BUCKET_NAME":
    raise ValueError("Please replace 'YOUR_PROJECT_ID' and 'YOUR_GCS_BUCKET_NAME' with your actual values.")

print(f"Using Project ID: {PROJECT_ID}")
print(f"Using GCS Bucket: {BUCKET_NAME}")

## 2. Incorporating `storage.py`

The `storage.py` module depends on `conidk.core.base.Auth` and `conidk.core.base.Config`. For this tutorial, we will create simple mock classes for these dependencies so that `storage.py` can be used directly without needing to install the `conidk` library.

We'll also use a `%%writefile` magic command to temporarily create `storage.py` and `conidk/core/base.py` files, allowing us to import them as modules.

In [ ]:
# Create mock conidk.core.base module
!mkdir -p conidk/core/
%%writefile conidk/core/base.py
"""Mock base module for demonstration."""
class Auth:
    """Mock Auth class."""
    def __init__(self):
        pass

class Config:
    """Mock Config class."""
    def __init__(self):
        pass
    def set_storage_endpoint(self):
        # In a real scenario, this might configure a specific GCS endpoint.
        # For this demo, we'll assume the default endpoint.
        return None

# Write the storage.py content to a file
%%writefile conidk/wrapper/storage.py
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""A wrapper for Google Cloud Storage (GCS) operations.

This module provides the `Gcs` class, which simplifies interactions with Google
Cloud Storage buckets. It allows for uploading, downloading, and listing files,
handling authentication and client setup.
"""

import enum
from typing import Optional, Union
from google.cloud.storage import Client #type: ignore

from conidk.core import base

class ContentType(enum.StrEnum):
    """Enum for supported content types."""
    TEXT = "text/plain"
    WAV = "audio/wav"

class Gcs:
    """A wrapper for Google Cloud Storage (GCS) operations.

    This class provides methods to interact with a GCS bucket, including
    uploading, downloading, and listing files. It simplifies client
    initialization and authentication.

    Attributes:
        bucket_name: The name of the GCS bucket.
        project_id: The Google Cloud project ID.
        auth: An authentication object.
        config: A configuration object.
        client: An instance of `google.cloud.storage.Client`.
    """

    def __init__(
        self,
        bucket_name: str,
        project_id: str,
        auth: Optional[base.Auth] = None,
        config: Optional[base.Config] = None,
    ) -> None:
        """Initializes the GCS wrapper.

        Args:
            bucket_name: The name of the GCS bucket (without 'gs://' prefix).
            project_id: The Google Cloud project ID.
            auth: An optional, pre-configured authentication object.
            config: An optional, pre-configured configuration object.
        """
        self.auth = auth or base.Auth()
        self.config = config or base.Config()

        self.bucket_name = bucket_name
        self.project_id = project_id
        self.client = Client(
            project=self.project_id,
            client_options=self.config.set_storage_endpoint(),
        )

    def download_blob(
        self,
        file_name: str,
        content_type: ContentType = ContentType.TEXT
    ) -> Union[str, bytes]:
        """Downloads a blob from the bucket.

        Args:
            file_name: The full path name of the blob to download.
            content_type: The expected content type of the blob.

        Returns:
            The contents of the blob as a decoded string or raw bytes.
        """

        bucket = self.client.bucket(self.bucket_name)
        blob = bucket.blob(file_name)
        if content_type == ContentType.WAV:
            return blob.download_as_bytes()
        return blob.download_as_string().decode("utf-8")

    def list_bucket(self) -> list[str]:
        """Lists blob names in the bucket.

        Returns:
            A list of the names of the blob objects.
        """

        tmp = []
        blobs = self.client.list_blobs(self.bucket_name)
        for blob in blobs:
            tmp.append(blob.name)
        return tmp

    def upload_blob(
            self,
            file_name: str,
            data: Union[str, bytes],
            content_type: ContentType = ContentType.TEXT
    ):
        """Uploads data to a blob in the bucket.

        Args:
            file_name: The name of the blob to upload to.
            data: The data to upload, either as a string or bytes.
            content_type: Optional content type for the blob.
        """

        bucket = self.client.bucket(self.bucket_name)
        blob = bucket.blob(file_name)
        if isinstance(data, str):
            blob.upload_from_string(data, content_type=content_type)
        else:
            blob.upload_from_file(data, content_type=content_type)


Now we can import the `Gcs` class and `ContentType` enum from our `storage` module.

In [ ]:
from conidk.wrapper.storage import Gcs, ContentType
import os # For file system operations if needed for cleanup or local files

## 3. Initializing the `Gcs` Client

The `Gcs` class constructor requires your `bucket_name` and `project_id`. It also accepts optional `Auth` and `Config` objects, which are handled by our mock classes in this demo.

In [ ]:
# Initialize the Gcs client
gcs_client = Gcs(
    bucket_name=BUCKET_NAME,
    project_id=PROJECT_ID
)

print(f"Gcs client initialized for bucket '{gcs_client.bucket_name}' in project '{gcs_client.project_id}'.")

## 4. Understanding `ContentType`

The `ContentType` enum defines common MIME types that can be specified when uploading blobs, helping GCS and other services correctly interpret the file type. The `storage.py` module supports `TEXT` (text/plain) and `WAV` (audio/wav).

In [ ]:
print(f"Text content type: {ContentType.TEXT}")
print(f"WAV content type: {ContentType.WAV}")

## 5. Uploading Data (`upload_blob`)

The `upload_blob` method allows you to upload string or bytes data to a specified file name within your GCS bucket. You can optionally set the `content_type`.

In [ ]:
# Example 1: Uploading a simple text string
text_file_name = "my_text_file.txt"
text_data = "Hello, GCS! This is a test string uploaded from the storage.py tutorial."

gcs_client.upload_blob(text_file_name, text_data, content_type=ContentType.TEXT)
print(f"Uploaded '{text_file_name}' as text/plain.")

# Example 2: Uploading binary data (e.g., a dummy WAV file)
wav_file_name = "dummy_audio.wav"
dummy_wav_data = b'RIFF\x00\x00\x00\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00data\x00\x00\x00\x00' # A minimal, silent WAV header

gcs_client.upload_blob(wav_file_name, dummy_wav_data, content_type=ContentType.WAV)
print(f"Uploaded '{wav_file_name}' as audio/wav.")

## 6. Listing Bucket Contents (`list_bucket`)

The `list_bucket` method retrieves the names of all objects currently stored in your configured GCS bucket.

In [ ]:
print(f"Listing blobs in bucket '{BUCKET_NAME}':")
blobs_in_bucket = gcs_client.list_bucket()

if blobs_in_bucket:
    for blob_name in blobs_in_bucket:
        print(f"- {blob_name}")
else:
    print("No blobs found in the bucket.")

## 7. Downloading Data (`download_blob`)
The `download_blob` method fetches the content of a specified object from your GCS bucket. By providing the `content_type`, you can control whether the data is returned as a decoded string (for text) or as raw bytes (for binary files like audio).

In [ ]:
# Download the text file we uploaded earlier
downloaded_content = gcs_client.download_blob(text_file_name, content_type=ContentType.TEXT)

print(f"Downloaded content from '{text_file_name}':")
print("----------------------------------------")
print(downloaded_content)
print("----------------------------------------")

# Now, download the WAV file as bytes
downloaded_wav_bytes = gcs_client.download_blob(wav_file_name, content_type=ContentType.WAV)

print(f"Downloaded content from '{wav_file_name}':")
print(f"Type of downloaded data: {type(downloaded_wav_bytes)}")
print(f"Downloaded {len(downloaded_wav_bytes)} bytes.")
print(f"First 20 bytes: {downloaded_wav_bytes[:20]}...")

# Verify the downloaded bytes match the uploaded dummy data
assert downloaded_wav_bytes == dummy_wav_data
print("\nSuccessfully verified that the downloaded WAV bytes match the uploaded data.")

## 8. Cleanup (Optional)

While `storage.py` does not include a `delete_blob` method, you can easily delete the uploaded files using the underlying `google.cloud.storage` client if you wish to clean up your bucket.

In [ ]:
from google.cloud import storage

print("Cleaning up uploaded files...")

client = storage.Client(project=PROJECT_ID)
bucket = client.get_bucket(BUCKET_NAME)

for file_to_delete in [text_file_name, wav_file_name]:
    blob = bucket.blob(file_to_delete)
    if blob.exists():
        blob.delete()
        print(f"Deleted '{file_to_delete}'.")
    else:
        print(f"'{file_to_delete}' not found, skipping deletion.")

print("Cleanup complete.")

# Verify cleanup
print(f"\nRemaining blobs in '{BUCKET_NAME}': {gcs_client.list_bucket()}")

## Conclusion

This tutorial demonstrated how to set up, initialize, and use the `storage.py` module to interact with Google Cloud Storage. You've learned how to:

*   Authenticate your environment for GCS access.
*   Initialize the `Gcs` client with your project and bucket details.
*   Understand and use the `ContentType` enum.
*   Upload both text and binary data using `upload_blob`.
*   List the contents of your bucket using `list_bucket`.
*   Download blob content using `download_blob`.

`storage.py` provides a simplified interface for common GCS tasks, making it easier to integrate cloud storage into your Python applications.